In [1]:
import os
import numpy as np
from tqdm import tqdm

In [2]:
DATA_DIR = "./data/raw"
OUT_DIR = "./data/processed"
os.makedirs(OUT_DIR, exist_ok=True)


DELTA_PHASE_Cb = 0.0360
DELTA_PHASE_B = 0.0410
DELTA_EPS = 1e-6
DOWNSAMPLE = 0

In [3]:
def parse_nso(line: str):
    """
    Parse a global-observable line starting with '# Nso'.

    Example input line:
        # Nso 16344 104277 253668 110490 100028 1.163670 3740

    According to the CDT data format, we:
      - take all numeric values after 'Nso'
      - drop the last two values
      - keep the very last value
    This selects the subset of global observables used in the paper.

    Returns:
        List[float]: selected global observables
    """
    # split line into tokens
    parts = line.split()

    # find the position of the 'Nso' keyword
    idx = parts.index("Nso")

    # take everything after 'Nso'
    after = parts[idx + 1:]

    # keep all but the last two entries, and also keep the final entry
    values = after[:-2] + after[-1:]

    # convert all values to float
    return [float(v) for v in values]


def parse_vto(line: str):
    """
    Parse a local-observable line starting with 'Vto'.

    Example input line:
        Vto t x1 x2 x3 x4 x5 x6

    The first two entries ('Vto' and time index t) are discarded.
    Only the six local geometric observables are returned.

    Returns:
        List[float]: local observables for a single time slice
    """
    # split line into tokens
    parts = line.split()

    # skip 'Vto' and time index, keep x1..x6
    return [float(v) for v in parts[2:]]


def parse_delta_from_filename(filename: str) -> float:
    return float(filename.split("-")[2])*1


def flatten_sample(sample):
    """
    Convert a single CDT configuration into a flat feature vector.

    The feature vector consists of:
      - global observables (Nso)
      - local observables (Vto), ordered by discrete time slice

    This ordering enforces time-translation symmetry after
    cyclic time-shift augmentation.

    Returns:
        List[float]: 1D feature vector (length = 30)
    """
    features = []

    # add global observables
    features.extend(sample["Nso"])

    # add local observables in time order
    for vto_t in sample["Vto"]:
        features.extend(vto_t)

    return features

def label_from_delta(delta):
    """
    Assign a phase label based on delta.

    Deep inside phase A:
        label = 0
    Deep inside phase B:
        label = 1
    Intermediate delta values:
        label = None (not used for training)

    A small tolerance is used to avoid floating-point issues.

    Returns:
        int or None
    """
    if abs(delta - DELTA_PHASE_Cb) < DELTA_EPS:
        return 0
    if abs(delta - DELTA_PHASE_B) < DELTA_EPS:
        return 1
    return None

In [4]:
files = sorted(
    f for f in os.listdir(DATA_DIR)
    if f.startswith("vto-") and f.endswith(".out")
)

In [5]:
for file in files:
    test = parse_delta_from_filename(file)
    print(test)

0.036
0.0366
0.037
0.0374
0.0378
0.0382
0.0386
0.039
0.0394
0.0398
0.0402
0.041


In [6]:
def process_sample(
    current_sample,
    sample_counter,
    skip_samples,
    file_delta,
    file_label,
):
    if sample_counter <= skip_samples:
        return

    assert len(current_sample["Vto"]) == 4
    assert current_sample["Nso"] is not None
    
    # X_shifts[shift].append(flatten_sample(perm))
    # y_shifts[shift].append(file_label)
    X_full.append(flatten_sample(current_sample))
    Delta_full.append(file_delta)
    y_full.append(file_label)


In [7]:
# Containers for the final dataset (filled after concatenation)
X_full = []
y_full = []
Delta_full = []

# Separate buffers for each time-shift variant
# # shift = 0, 1, 2, 3 correspond to cyclic time translations
# X_shifts = [[], [], [], []]
# y_shifts = [[], [], [], []]
# Delta_shifts = [[], [], [], []]


# Loop over all CDT output files (each file corresponds to a fixed delta)
for file_idx, filename in enumerate(tqdm(files, desc="Parsing files")):

    # Number of initial Monte Carlo configurations to discard
    # (thermalization cut; endpoints use a slightly smaller cut)
    # SKIP_SAMPLES = 2000 if file_idx in (0, len(files) - 1) else 2200
    SKIP_SAMPLES = 0
    # Extract delta value from filename and assign phase label (if deep A or B)
    file_delta = parse_delta_from_filename(filename)
    file_label = label_from_delta(file_delta)

    file_path = os.path.join(DATA_DIR, filename)

    # Temporary storage for the currently parsed configuration
    current_sample = None
    sample_counter = 0

    # Read the file line by line
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            # Marker for the beginning of a new configuration
            if line.startswith("# ntime") or line.startswith("SF"):

                # If a previous configuration was fully read, process it
                if current_sample is not None:
                            sample_counter += 1
                        
                            process_sample(
                                current_sample,
                                sample_counter,
                                SKIP_SAMPLES,
                                file_delta,
                                file_label,
                            )
                # Initialize a new configuration container
                current_sample = {
                    "Nso": None,   # global observables
                    "Vto": []      # list of local observables (one per time slice)
                }

            # Parse global observables
            elif line.startswith("# Nso") or line.startswith("Nso"):
                current_sample["Nso"] = parse_nso(line)

            # Parse local observables for a single time slice
            elif line.startswith("Vto"):
                current_sample["Vto"].append(parse_vto(line))


    if current_sample is not None:
                sample_counter += 1
            
                process_sample(
                    current_sample,
                    sample_counter,
                    SKIP_SAMPLES,
                    file_delta,
                    file_label,
                )
    # Report number of equilibrated configurations processed in this file
    print(f"{filename}: parsed {sample_counter - SKIP_SAMPLES} samples")

    
X_full = np.asarray(X_full)
y_full = np.asarray(y_full, dtype=object)
Delta_full = np.asarray(Delta_full)



Parsing files:   8%|██▍                          | 1/12 [00:02<00:23,  2.18s/it]

vto-2.2-0.0360-100k-T4-torus.out-L.out: parsed 112739 samples


Parsing files:  17%|████▊                        | 2/12 [00:04<00:20,  2.04s/it]

vto-2.2-0.0366-100k-T4-torus.out: parsed 100000 samples


Parsing files:  25%|███████▎                     | 3/12 [00:05<00:17,  1.95s/it]

...

vto-2.2-0.0398-100k-T4-torus.out: parsed 95805 samples


Parsing files:  92%|█████████████████████████▋  | 11/12 [00:22<00:01,  1.98s/it]

vto-2.2-0.0402-100k-T4-torus.out: parsed 90065 samples


Parsing files: 100%|████████████████████████████| 12/12 [00:23<00:00,  2.00s/it]

vto-2.2-0.0410-100k-T4-torus.out-L.out: parsed 128040 samples


In [8]:
assert X_full.shape[1] == 30

mask_train = (
    (Delta_full == DELTA_PHASE_Cb) |
    (Delta_full == DELTA_PHASE_B)
) & (y_full != None)

X_train_aug = [[], [], [], []]
y_train_aug = [[], [], [], []]
Delta_train_aug = [[], [], [], []]

for x, y, delta in zip(
    X_full[mask_train],
    y_full[mask_train],
    Delta_full[mask_train]
):
    nso = x[:6]
    vto = x[6:].reshape(4, 6)

    for shift in range(4):
        vto_shift = np.roll(
            vto,
            -shift,
            axis=0
        )

        x_shift = np.concatenate([
            nso,
            vto_shift.flatten()
        ])

        X_train_aug[shift].append(x_shift)
        y_train_aug[shift].append(y)
        Delta_train_aug[shift].append(delta)


X_train_aug = np.concatenate(
    [np.asarray(X_train_aug[s]) for s in range(4)],
    axis=0
)

y_train_aug = np.concatenate(
    [np.asarray(y_train_aug[s]) for s in range(4)],
    axis=0
)

Delta_train_aug = np.concatenate(
    [np.asarray(Delta_train_aug[s]) for s in range(4)],
    axis=0
)

        

assert len(X_train_aug) == 4 * np.sum(mask_train)
print("Full samples:", len(X_full))
print("Train samples:", np.sum(mask_train))
print("Train augmented:", len(X_train_aug))


Full samples: 1334615
Train samples: 240779
Train augmented: 963116


In [9]:
X_full = np.asarray(X_full, dtype=np.int64)

X_train_aug = np.asarray(X_train_aug, dtype=np.int64)
y_train_aug = np.asarray(y_train_aug)
Delta_train_aug = np.asarray(Delta_train_aug)

np.savez(
    os.path.join(OUT_DIR, "FULL_30.npz"),
    X=X_full,
    y=y_full,
    Delta=Delta_full,
)

print("FULL_30.npz zapisany")
print("X_full shape:", X_full.shape)

np.savez(
    os.path.join(OUT_DIR, "TRAIN_30.npz"),
    X=X_train_aug,
    y=y_train_aug,
    Delta=Delta_train_aug,
)

print("TRAIN_30.npz zapisany")
print("X_train shape:", X_train_aug.shape)

FULL_30.npz zapisany
X_full shape: (1334615, 30)
TRAIN_30.npz zapisany
X_train shape: (963116, 30)
